# 单手牌解析分析工具

本 Notebook 用于解析单个手牌历史文件，生成 `HandHistory` 和 `State` 对象，并打印详细信息。

In [1]:
# 导入依赖
from pathlib import Path
from bayes_poker.hand_history.parse_gg_poker import (
    RushCashPokerStarsParser,
    parse_value_in_cents,
    sanitize_hand_text,
)
from pokerkit import NoLimitTexasHoldem

## 1. 输入手牌文件路径

修改下方 `hand_path` 变量，指向你要分析的 `.txt` 文件。

In [2]:
# 配置: 输入手牌文件路径
hand_path = Path("../../data/sample_hand.txt")  # <- 修改为你的手牌文件路径

# 或直接粘贴手牌文本
hand_text_raw = """
PokerStars Hand #03233240636: Hold'em No Limit ($0.01/$0.02) - 2025/01/12 06:25:03
Table 'GG_RushAndCash1694942' 6-max Seat #1 is the button
Seat 1: wachang ($2.00 in chips)
Seat 2: Juandiecv ($1.28 in chips)
Seat 3: SlavyaniusP ($2.00 in chips)
Seat 4: FuLuShou poker ($4.12 in chips)
Seat 5: LanaLight ($2.14 in chips)
Seat 6: gravesJG ($2.07 in chips)
Cash Drop to Pot : total $0.20
Juandiecv: posts small blind $0.01
SlavyaniusP: posts big blind $0.02
*** HOLE CARDS ***
FuLuShou poker: raises $0.02 to $0.04
LanaLight: folds
gravesJG: calls $0.04
wachang: calls $0.04
Juandiecv: raises $0.38 to $0.42
SlavyaniusP: raises $1.58 to $2.00 and is all-in
FuLuShou poker: folds
gravesJG: folds
wachang: folds
Juandiecv: calls $0.86 and is all-in
Uncalled bet ($0.72) returned to SlavyaniusP
SlavyaniusP: shows [4c Jc]
Juandiecv: shows [Qh Jh]
*** FLOP *** [Qc 8s 7c]
Juandiecv: Chooses to EV Cashout
*** TURN *** [Qc 8s 7c] [8h]
*** RIVER *** [Qc 8s 7c 8h] [4h]
Juandiecv: Pays Cashout Risk ($1.07)
*** SHOWDOWN *** 
Juandiecv collected $2.79 from pot
*** SUMMARY ***
Total pot $2.68 | Rake $0.06 | Jackpot $0.03 | Bingo $0 | Fortune $0 | Tax $0
Board [Qc 8s 7c 8h 4h]
Seat 1: wachang (button) folded before Flop
Seat 2: Juandiecv (small blind) showed [Qh Jh] and won ($2.79), Cashout Risk ($1.07)
Seat 3: SlavyaniusP (big blind) showed [4c Jc] and lost
Seat 4: FuLuShou poker folded before Flop
Seat 5: LanaLight folded before Flop (didn't bet)
Seat 6: gravesJG folded before Flop
""".strip()

## 2. 加载并清洗手牌文本

In [3]:
# 从文件读取或使用直接粘贴的文本
if hand_path.exists():
    hand_text = hand_path.read_text(encoding="utf-8")
    print(f"从文件读取: {hand_path}")
else:
    hand_text = hand_text_raw
    print("使用直接粘贴的手牌文本")

# 清洗文本 (移除非标准行、处理 Cash Drop 等)
sanitized_text = sanitize_hand_text(hand_text)

print("\n--- 清洗后的手牌文本 ---")
print(sanitized_text)

使用直接粘贴的手牌文本

--- 清洗后的手牌文本 ---
PokerStars Hand #03233240636: Hold'em No Limit ($0.01/$0.02) - 2025/01/12 06:25:03
Table 'GG_RushAndCash1694942' 6-max Seat #1 is the button
Seat 1: wachang ($2.00 in chips)
Seat 2: Juandiecv ($1.28 in chips)
Seat 3: SlavyaniusP ($2.00 in chips)
Seat 4: FuLuShou poker ($4.12 in chips)
Seat 5: LanaLight ($2.14 in chips)
Seat 6: gravesJG ($2.07 in chips)
Cash Drop to Pot : total $0.20
Juandiecv: posts small blind $0.01
SlavyaniusP: posts big blind $0.02
*** HOLE CARDS ***
FuLuShou poker: raises $0.02 to $0.04
LanaLight: folds
gravesJG: calls $0.04
wachang: calls $0.04
Juandiecv: raises $0.38 to $0.42
SlavyaniusP: raises $1.58 to $2.00 and is all-in
FuLuShou poker: folds
gravesJG: folds
wachang: folds
Juandiecv: calls $0.86 and is all-in
Uncalled bet ($0.72) returned to SlavyaniusP
SlavyaniusP: shows [4c Jc]
Juandiecv: shows [Qh Jh]
*** FLOP *** [Qc 8s 7c]
*** TURN *** [Qc 8s 7c] [8h]
*** RIVER *** [Qc 8s 7c 8h] [4h]
*** SHOWDOWN *** 
Juandiecv collected $2.7

## 3. 解析为 HandHistory 对象

In [4]:
parser = RushCashPokerStarsParser()

try:
    hh = parser._parse(sanitized_text, parse_value=parse_value_in_cents)
    print("✅ 解析成功!")
except Exception as e:
    print(f"❌ 解析失败: {e}")
    hh = None

✅ 解析成功!


## 4. 打印 HandHistory 详情

In [5]:
if hh:
    print("=" * 60)
    print("HandHistory 属性")
    print("=" * 60)
    print(f"玩家列表: {hh.players}")
    print(f"起始筹码: {hh.starting_stacks}")
    print(f"最终奖金: {hh.winnings}")
    print(f"抽水 (Rake): {hh.rake}")
    print(f"\n动作序列 (前20条):")
    for i, action in enumerate(hh.actions[:20]):
        print(f"  {i}: {action}")
    if len(hh.actions) > 20:
        print(f"  ... (共 {len(hh.actions)} 条动作)")

HandHistory 属性
玩家列表: ['Juandiecv', 'SlavyaniusP', 'FuLuShou poker', 'LanaLight', 'gravesJG', 'wachang']
起始筹码: [128, 200, 412, 214, 207, 200]
最终奖金: [279, 0, 0, 0, 0, 0]
抽水 (Rake): functools.partial(<function rake at 0x72e0443945e0>, percentage=0)

动作序列 (前20条):
  0: d dh p1 ????
  1: d dh p2 ????
  2: d dh p3 ????
  3: d dh p4 ????
  4: d dh p5 ????
  5: d dh p6 ????
  6: p3 cbr 4
  7: p4 f
  8: p5 cc
  9: p6 cc
  10: p1 cbr 42
  11: p2 cbr 200
  12: p3 f
  13: p5 f
  14: p6 f
  15: p1 cc
  16: p2 sm ????
  17: p1 sm ????
  18: d db Qc8s7c
  19: p1 sm ????
  ... (共 27 条动作)


## 5. 生成并重放 State 对象

In [7]:
if hh:
    print("=" * 60)
    print("重放 State 对象")
    print("=" * 60)
    
    # 创建初始 State
    state = hh.create_state()
    
    print(f"\n初始状态:")
    print(f"  盲注结构: SB={state.blinds_or_straddles}")
    print(f"  按钮位置: {state.button_index}")
    print(f"  玩家筹码: {state.stacks}")
    
    # 重放所有动作
    print(f"\n重放动作:")
    for i, action in enumerate(hh.actions):
        try:
            # 解析并执行动作
            state(action)
            # 打印关键状态变化
            if i < 10 or "d db" in str(action):  # 只打印前10条或发牌
                print(f"  [{i}] {action}")
                print(f"       底池: {state.total_pot_amount}, 筹码: {state.stacks}")
        except Exception as e:
            print(f"  [{i}] ❌ 执行失败: {action} -> {e}")
            break
    
    print(f"\n最终状态:")
    print(f"  底池: {state.total_pot_amount}")
    print(f"  筹码: {state.stacks}")
    print(f"  状态完成: {state.status.name if hasattr(state, 'status') else 'N/A'}")

重放 State 对象

初始状态:
  盲注结构: SB=(1, 2, 0, 0, 0, 0)


AttributeError: 'State' object has no attribute 'button_index'

## 6. 可视化 (可选)

你可以在这里添加更多可视化代码，例如绘制筹码变化图等。

In [9]:
# 示例: 打印公共牌
if hh:
    board_actions = [str(a) for a in hh.actions if "d db" in str(a)]
    if board_actions:
        print("公共牌发牌动作:")
        for b in board_actions:
            print(f"  {b}")
    else:
        print("未发现公共牌 (可能在 Flop 前结束)")

公共牌发牌动作:
  d db 8h8cJh
  d db Qc
